# Working with atmoz.lidar.TOLNet

**Author:** Maurice Roots  
**Contact:** mrmoroots@gmail.com  
For more information on TOLNet, visit [https://tolnet.larc.nasa.gov/](https://tolnet.larc.nasa.gov/)

---

## Summary

This notebook shows how to use the `atmoz.lidar.TOLNET` class to query, download, and visualize vertical ozone profiles from the Tropospheric Ozone Lidar Network (TOLNet). You will learn how to read curtain plots, filter data by instrument and product type, and extract quantitative information for your own research.

## Table of Contents

1. [Dataset Information](#section1)
2. [Data Acquisition](#section2)
3. [Exploring the Data Structure](#section3)
4. [Curtain Plots](#section4)
5. [Case Study: 2025 Ozone Episode](#section5)
6. [Appendix: Lidar Locations](#appendix)

## Prerequisites

```
pip install git+https://github.com/moroots/atmoz
```


## Introduction: Lidar vs. Surface Ozone Measurement

**Surface monitors** (like those in the AQS network) measure ozone at a single altitude — typically 3–10 m above the ground. They tell you what the air quality is at breathing level, but nothing about how ozone is distributed vertically through the atmosphere.

**Lidar** (Light Detection And Ranging) instruments fire short laser pulses upward and measure the backscattered light at different altitudes. An ozone differential absorption lidar (DIAL) can resolve the full vertical ozone profile from near the ground up to ~15 km altitude, continuously in time. This vertical information is critical for understanding:

- How ozone from the stratosphere intrudes into the troposphere on certain weather patterns
- How urban pollution accumulates within the planetary boundary layer (~0–2 km)
- How wildfire smoke or long-range transport events are distributed aloft

### What is a Curtain Plot?

A **curtain plot** (also called a time-altitude cross-section) is the primary visualization for lidar data:

| Axis | Meaning |
|------|---------|
| **x-axis** | Time (UTC) |
| **y-axis** | Altitude above ground (km) |
| **Color** | Ozone mixing ratio (ppbv — parts per billion by volume) |

Dark red and gray colors indicate high ozone (> ~80–100 ppbv). The **EPA NAAQS threshold for surface ozone is 70 ppb** (based on the daily maximum 8-hour average), but lidar data show the full vertical structure — ozone often peaks well above the surface during certain events.

### What is TOLNet?

The **Tropospheric Ozone Lidar Network (TOLNet)** is an interagency network co-managed by NASA, NOAA, and EPA, initiated in 2011. It coordinates ozone lidar observations from multiple research groups across North America and provides data through a public API at [https://tolnet.larc.nasa.gov/](https://tolnet.larc.nasa.gov/).

In [ ]:
from atmoz.lidar import TOLNET
import pandas as pd
import matplotlib.pyplot as plt

## 1. Dataset Information <a class="anchor" id="section1"></a>

The Tropospheric Ozone Lidar Network (TOLNet) is an interagency network initiated by NASA, NOAA, and EPA in 2011 with the goal of enhancing tropospheric ozone measurement capabilities. Ozone lidars within TOLNet provide accurate observations under both daytime and nighttime conditions and generate consistent, long-term datasets. Most instruments are portable and have been deployed in air quality campaigns in coordination with federal, state, and local agencies.

The data is stored as HDF4 files, but the TOLNet API returns JSON data, which is what this package uses.

Charter lidar systems are affiliated with NASA GSFC, NASA LaRC, NASA Jet Propulsion Laboratory (JPL), NOAA ESRL/CSL, and the University of Alabama, Huntsville. Systems have also been developed at CCNY and Hampton University. An international collaboration with Environment and Climate Change Canada (ECCC) has been established, with modeling support at NASA Ames and data archiving at NASA LaRC.

> **A note on data use:** Consulting with the instrument principal investigator or co-investigators before publishing results is strongly encouraged. DOI and citation links are provided in the `instrument_groups` table below.

### 1a. Instrument Groups

Each instrument in TOLNet belongs to an **instrument group** identified by an integer ID. Use these IDs when filtering queries. The `home_location` column shows where the instrument is typically stationed, though mobile instruments can be deployed elsewhere.

In [ ]:
tolnet = TOLNET()
tolnet.instrument_groups

### 1b. File Types

TOLNet hosts multiple file formats per observation. The `import_data()` method uses the API's JSON endpoint (`/data/json/{file_id}`), which is only available for **HDF GEOMS** files (`file_type_id = 1`). Files of other types (images, ASCII, raw) will fail silently and be counted in `self._errors` — this is expected behavior, not a bug.

In [ ]:
tolnet.file_types

## 2. Data Acquisition <a class="anchor" id="section2"></a>

### Query Parameters

`import_data()` accepts the following filter parameters. All are optional except `min_date` and `max_date`.

| Parameter | Type | Description |
|-----------|------|-------------|
| `min_date` | str | Start date `YYYY-MM-DD` (required) |
| `max_date` | str | End date `YYYY-MM-DD` (required) |
| `product_type` | list[int] | Recommend `[4]` (HIRES) — the only fully supported product |
| `instrument_group` | list[int] | Filter to specific instruments (IDs from table above) |
| `processing_type` | list[int] | Filter by processing method (1 = Centrally Processed) |
| `file_type` | list[int] | Filter by file format (leave empty to get all) |

> **Why `product_type=[4]` (HIRES)?** The high-resolution product has the finest altitude and time resolution and is the only type currently supported by the JSON conversion endpoint.

In [ ]:
# --- Configure your query ---
DATE_START   = "2025-08-08"
DATE_END     = "2025-08-11"
PRODUCT_TYPE = [4]           # HIRES — recommended

tolnet.import_data(min_date=DATE_START, max_date=DATE_END, product_type=PRODUCT_TYPE)

`import_data()` downloads and parses each matching file from the API, then stores the results in:

- `tolnet.data` — `Dict[(group, processing_type, lat_lon) → Dict[date → DataFrame]]`  
- `tolnet.meta_data` — same key structure, values are raw metadata dicts  
- `tolnet._errors` — list of files that could not be parsed (expected for non-HDF types)

The data is **not** returned — it is stored on the object. Access it via `tolnet.data` as shown in the next section.

## 3. Exploring the Data Structure <a class="anchor" id="section3"></a>

### 3a. Data Keys

`tolnet.data` is a nested dictionary. The **outer keys** are 3-tuples:

```
(instrument_group_name, processing_type_name, "latitude x longitude")
```

Each key uniquely identifies one instrument at one location processed one way. If you query multiple instruments, you get multiple keys.

In [ ]:
# Show all (instrument, processing_type, location) combinations in this query
list(tolnet.data.keys())

### 3b. Date Keys and DataFrames

Within each outer key, the inner keys are **date strings** (`"YYYY-MM-DD"`). Each value is a `pandas.DataFrame` where:

- **index** = UTC timestamps (one row per lidar measurement)  
- **columns** = altitude above ground in **km**  
- **values** = ozone mixing ratio in **ppbv**

In [ ]:
# Pick one key to explore
key  = list(tolnet.data.keys())[0]
date = list(tolnet.data[key].keys())[0]

print(f"Key:  {key}")
print(f"Date: {date}")
print()

df_sample = tolnet.data[key][date]
print(f"Shape: {df_sample.shape}  (rows=timestamps, cols=altitudes)")
print(f"Altitude range: {float(df_sample.columns.min()):.2f} – {float(df_sample.columns.max()):.2f} km")
df_sample.head()

### 3c. Metadata

In [ ]:
# View metadata for the same file — use .copy() to avoid mutating the stored dict
meta = tolnet.meta_data[key][list(tolnet.meta_data[key].keys())[0]].copy()

# Strip the large data arrays so we only print the descriptive fields
for field in ("data", "value", "altitude", "datetime"):
    if field in meta and isinstance(meta[field], dict) and "data" in meta[field]:
        meta[field] = {k: v for k, v in meta[field].items() if k != "data"}

meta

## 4. Curtain Plots <a class="anchor" id="section4"></a>

### How to Read a Curtain Plot

```
  altitude (km)
  ^
8 |
  |     ░░░░░░░░░░░░░░░░░░
4 |   ████████████████████
  | ░░░░░░░░░████████░░░░░░
2 |___________________________> time (UTC)
```

- **Color scale:** pale yellow → orange → red → gray = increasing ozone ppbv  
- **Boundary layer** (0–2 km): ozone here reflects local photochemistry and urban emissions  
- **Free troposphere** (2–8 km): baseline ozone; enhancements can indicate long-range transport  
- **Stratospheric intrusions**: sudden high-ozone tongues reaching down from >8 km  
- **EPA NAAQS context:** the surface standard is **70 ppb**; lidar captures how that boundary-layer ozone relates to the full column

### 4a. Plot All Instruments

In [ ]:
# One curtain plot per (instrument, processing_type, location) key
for key, item in tolnet.data.items():
    tolnet.plot_curtain(item)

### 4b. Filtered Query

Filter to a single instrument and use `tolnet_curtains()` as a convenient shorthand that plots all keys at once.

In [ ]:
# --- Change INSTRUMENT_GROUP to any ID from the instrument_groups table ---
INSTRUMENT_GROUP = [7]   # NASA JPL SMOL-1

params = {
    "min_date":         DATE_START,
    "max_date":         DATE_END,
    "product_type":     PRODUCT_TYPE,
    "instrument_group": INSTRUMENT_GROUP,
}
data = tolnet.import_data(**params)

In [ ]:
# Plot all keys in the filtered result
data.tolnet_curtains()

### 4c. Customizing the Plot Window

In [ ]:
# xlims: crop the time axis to a sub-range (ISO 8601 date strings)
data.tolnet_curtains(xlims=["2025-08-08", "2025-08-09"], title="Cropped — Time Axis")

In [ ]:
# ylims: crop the altitude axis (units: km)
data.tolnet_curtains(ylims=[0, 4], title="Cropped — Altitude Axis (0–4 km)")

In [ ]:
# Custom axis labels
data.tolnet_curtains(xlabel="Time (UTC)", ylabel="Height AGL (km)", title="Custom Labels")

## 5. Case Study: 2025 Ozone Episode <a class="anchor" id="section5"></a>

The summer of 2025 produced several elevated ozone events in the eastern U.S. driven by a combination of heat, stagnant air masses, and regional transport. Below we set up a targeted query to examine one such event in detail.

**How to use this section:** The parameters in the next cell are clearly labeled — swap them to explore any instrument, date range, or location from the `instrument_groups` table.

In [ ]:
# ── Case Study Configuration ─────────────────────────────────────────────────
CS_DATE_START       = "2025-08-08"
CS_DATE_END         = "2025-08-11"
CS_INSTRUMENT_GROUP = [2]          # NASA GSFC (Goddard Space Flight Center, MD)
CS_PRODUCT_TYPE     = [4]          # HIRES

# ── Fetch and plot ────────────────────────────────────────────────────────────
cs = TOLNET()
cs.import_data(
    min_date=CS_DATE_START,
    max_date=CS_DATE_END,
    instrument_group=CS_INSTRUMENT_GROUP,
    product_type=CS_PRODUCT_TYPE,
)
cs.tolnet_curtains()

### Interpreting the Case Study Curtain

In the curtain plot above, look for:

- **Elevated ozone layers** (orange/red bands) lifting off the surface into the lower free troposphere during daytime hours — a signature of convective mixing
- **Boundary layer deepening** across the day (the base of clean air rising as the mixed layer grows with solar heating)
- **Nighttime low ozone near the surface** — after sunset the boundary layer collapses, titrating ozone via NO emissions from traffic

> The **EPA NAAQS for ozone is 70 ppb**. Any area in the curtain showing values above that threshold (warm colors near or above the surface) represents air quality that would be unhealthy for sensitive groups if sustained at ground level for 8 hours.

### 5a. Concatenate All Days into One DataFrame

In [ ]:
# The inner dict maps date string → DataFrame. Concatenate to get the full time series.
cs_key = list(cs.data.keys())[0]
df_cs = pd.concat(list(cs.data[cs_key].values())).sort_index()

group_name, proc_name, lat_lon = cs_key
print(f"Instrument:  {group_name}")
print(f"Processing:  {proc_name}")
print(f"Location:    {lat_lon}")
print(f"Time range:  {df_cs.index.min()} → {df_cs.index.max()}")
print(f"Altitude range: {float(df_cs.columns.min()):.2f} – {float(df_cs.columns.max()):.2f} km")
print(f"Shape:  {df_cs.shape}  (timestamps × altitude levels)")

### 5b. Vertical Ozone Profile at a Specific Time

A **vertical profile** is a single column of the curtain: the ozone mixing ratio at every altitude at one moment in time. Comparing profiles at different times reveals how the vertical ozone distribution evolves throughout the day.

In [ ]:
# Choose a time near local noon (peak photochemistry) — adjust as needed
profile_time = df_cs.index[len(df_cs) // 2]
profile = df_cs.loc[profile_time].dropna()

fig, ax = plt.subplots(figsize=(5, 8))
ax.plot(profile.values, profile.index.astype(float), color="#CD6091", linewidth=2.2)
ax.axvline(70, color="#E76F51", linestyle="--", linewidth=1.5, label="EPA NAAQS (70 ppb)")
ax.set_xlabel("Ozone (ppbv)", fontsize=12)
ax.set_ylabel("Altitude (km)", fontsize=12)
ax.set_title(f"Vertical Ozone Profile\n{profile_time.strftime('%Y-%m-%d %H:%M UTC')}",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

**What to look for in the profile:**
- A sharp rise in ozone near the top of the boundary layer (~1–2 km) indicates photochemical production within the mixed layer
- A secondary peak at 5–10 km can signal stratospheric ozone intrusion or long-range transport from distant pollution sources
- Near-zero ozone below ~0.5 km at night reflects titration by NO from fresh traffic emissions

### 5c. Altitude of Peak Ozone Per Day

Tracking the altitude at which ozone is highest each day can reveal the dominant ozone source:

- **Low peak altitude (0–2 km):** local photochemistry / boundary layer production
- **High peak altitude (4–12 km):** stratospheric intrusion or long-range transport aloft

In [ ]:
# At each timestamp, find the altitude column with the highest ozone value
peak_alt = df_cs.idxmax(axis=1).astype(float)   # Series: timestamp → altitude (km)

# Daily median of the peak altitude
daily_peak_alt = peak_alt.resample("1D").median().dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily_peak_alt.index, daily_peak_alt.values,
        color="#7195AB", linewidth=2.2, marker="o", markersize=7, label="Daily median")
ax.set_xlabel("Date (UTC)", fontsize=12)
ax.set_ylabel("Altitude of Peak O₃ (km)", fontsize=12)
ax.set_title(f"Daily Altitude of Maximum Ozone — {group_name}", fontsize=13, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=11)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Appendix: Table of Lidar Locations <a class="anchor" id="appendix"></a>

This table lists known deployment locations for TOLNet instruments. Mobile instruments may appear at multiple sites; stationary instruments have a single routine location.

| Location | Mode | Instrument Group | Date Range | Latitude | Longitude |
|---|---|---|---|---|---|
| Albuquerque Fiesta Park NM | Campaign | NASA JPL SMOL-1 | 2023-10-11 – 2023-10-15 | 35.19 | −106.56 |
| Cabauw | Campaign | NASA JPL SMOL-1 | 2024-05-22 – 2024-06-09 | 51.97 | 4.93 |
| Cabauw | Campaign | NASA GSFC | 2019-09-12 – 2019-10-02 | 51.97 | 4.93 |
| Fort Mackay | Campaign | ECCC | 2016-11-04 – 2019-09-23 | 57.19 | −111.62 |
| Guilford YCFS CT | Campaign | NOAA ESRL/CSL | 2023-07-04 – 2023-08-14 | 41.25 | −72.75 |
| Huntsville AL | Campaign | UAH | 2022-07-08 – 2022-07-10 | 30.27 | −88.12 |
| Kenosha WI | Campaign | UAH | 2023-07-18 – 2023-08-16 | 42.50 | −87.81 |
| La Porte TX | Campaign | NASA GSFC | 2021-08-11 – 2021-09-28 | 29.67 | −95.06 |
| Langley Research Center VA | Campaign | NASA LaRC | 2018-01-24 – 2024-07-05 | 37.09 | −76.38 |
| Pasadena JPL CA | Campaign | NASA JPL SMOL-1 | 2023-06-25 – 2023-09-07 | 34.19 | −118.19 |
| San Bernardino Calstate CA | Campaign | NASA JPL SMOL-2 | 2023-06-23 – 2023-12-16 | 34.19 | −117.31 |
| Sherwood Island CT | Campaign | NASA LaRC | 2018-07-12 – 2023-08-26 | 41.12 | −73.31 |
| University of Houston Moody Tower TX | Campaign | NASA LaRC | 2021-08-26 – 2021-09-27 | 29.72 | −95.31 |
| Beltsville MD | Routine | NASA GSFC | 2015-06-10 – 2022-07-28 | 39.06 | −76.88 |
| Boulder CO | Routine | NOAA ESRL/CSL | 2018-02-08 – 2023-10-23 | 40.00 | −105.25 |
| Goddard Space Flight Center MD | Routine | NASA GSFC | 2015-02-03 – 2024-06-25 | 38.97 | −76.81 |
| Huntsville AL | Routine | UAH | 2017-07-31 – 2023-04-20 | 34.72 | −86.62 |
| New York NY | Routine | CCNY | 2023-06-01 – 2024-07-11 | 40.81 | −73.94 |
| Table Mountain CA | Routine | NASA JPL TMTOL | 2000-01-04 – 2024-07-10 | 34.38 | −117.69 |

## Conclusion

In this tutorial we used `atmoz.lidar.TOLNET` to work with vertical ozone lidar data end-to-end:

- **Explored** the TOLNet network (13 instrument groups, multiple file types)  
- **Queried** the API with date, instrument, product type, and processing type filters  
- **Visualized** ozone as curtain plots and customized the time and altitude windows  
- **Extracted** quantitative information: vertical profiles at specific times and the daily altitude of peak ozone

### Suggested Next Steps

| Idea | How |
|------|-----|
| Explore a different instrument | Change `CS_INSTRUMENT_GROUP` in the case study config |
| Look at a different season | Adjust `CS_DATE_START` / `CS_DATE_END` to winter months for different chemistry |
| Compare two processing types | Query without `processing_type` filter; two keys will appear |
| Compare lidar with surface | Run the **surface AQS tutorial** (`atmoz_tutorial_surface.ipynb`) for the same dates |
| Model comparison | See the **GEOS-CF tutorial** (`atmoz_tutorial_geoscf.ipynb`) for co-located model profiles |